# 1挂载云盘

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

# 2安装依赖

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

!pip install faster-whisper pysrt pyannote.audio imageio-ffmpeg audio-separator[gpu] ctranslate2==4.4.0

!apt-get update
!apt-get install libcudnn8=8.9.2.26-1+cuda12.1
!apt-get install libcudnn8-dev=8.9.2.26-1+cuda12.1
!python -c "import torch; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True"
# Workaround from: https://github.com/m-bain/whisperX/issues/1027#issuecomment-2627525081

# 3导入配置

In [ ]:
import torch
asr_language="ja"
asr_model="4evergr8/whisper-large-v2-translate-zh-v0.2-st-ct2"
compute_type ="int8"
device = "cuda" if torch.cuda.is_available() else "cpu"
path_asr="3asr"
path_audio="1audio"
path_model="model"
path_pre="0pre"
path_vad="2vad"
sep_model="MDX23C-8KFFT-InstVoc_HQ_2.ckpt"
vad_min_duration_off=0.1
vad_min_duration_on=0.3
vad_model="4evergr8/pyannote-segmentation-3.0"

# 4人声分离

In [ ]:
import os
import shutil
import subprocess
from audio_separator.separator import Separator
#############################################################################################################################
for filename in os.listdir(path_pre):
    if filename.lower().endswith((".wav", ".mp3", ".flac")):
        basename = os.path.splitext(filename)[0]

        slice_path = os.path.join(path_pre, f"{basename}-slice")
        split_path = os.path.join(path_pre, f"{basename}-split")
        audio_path = os.path.join(path_pre, filename)

        if not os.path.isfile(audio_path):
            continue

        # ===== 切片 =====
        if os.path.exists(slice_path):
            shutil.rmtree(slice_path)
        os.makedirs(slice_path)

        segment_length = 1200

        command = [
            "ffmpeg",
            "-i", audio_path,
            "-f", "segment",
            "-segment_time", str(segment_length),
            "-c", "copy",
            os.path.join(slice_path, "%03d.wav")
        ]
        subprocess.run(command, check=True)

        # ===== 分离 =====
        if not os.path.exists(split_path):
            os.makedirs(split_path)

        separator = Separator(
            model_file_dir=path_model,
            output_dir=split_path,
            output_single_stem="vocals",
            sample_rate=16000,
        )
        separator.load_model(model_filename=sep_model)

        for file in os.listdir(slice_path):
            if file.endswith(".wav"):
                slice_basename = os.path.splitext(file)[0]

                exists = any(name.startswith(slice_basename) for name in os.listdir(split_path))
                if exists:
                    print(f"跳过: {file}")
                    continue

                separator.separate(os.path.join(slice_path, file))

        # ===== 拼接 =====
        file_list = sorted(
            [f for f in os.listdir(split_path) if f.endswith(".wav")],
            key=lambda x: int(x[:3])
        )

        list_path = os.path.join(path_pre, f"{basename}_list.txt")
        with open(list_path, "w", encoding="utf-8") as f:
            for f_name in file_list:
                full_path = os.path.join(split_path, f_name)
                f.write(f"file '{full_path}'\n")

        output_path = os.path.join(path_audio, f"{basename}.wav")

        command = [
            "ffmpeg",
            "-f", "concat",
            "-safe", "0",
            "-i", list_path,
            "-c", "copy",
            output_path
        ]
        subprocess.run(command, check=True)

        print(f"完成人声分离: {output_path}")




# 5人声识别

In [ ]:
import os
import pysrt
import torch
from pyannote.audio import Model
from pyannote.audio.pipelines import VoiceActivityDetection
#############################################################################################################################
print('设备:', device)
vad = Model.from_pretrained(checkpoint=vad_model, cache_dir=path_model)
vad.to(torch.device(device))
vad_pipeline = VoiceActivityDetection(segmentation=vad)
vad_pipeline.instantiate({
    "min_duration_on": vad_min_duration_on,
    "min_duration_off": vad_min_duration_off,
})

for filename in os.listdir(path_audio):
    if not filename.endswith((".wav", ".mp3", ".flac")):
        continue
    audio_path = os.path.join(path_audio, filename)
    print(f"\n处理音频: {audio_path}")

    basename = os.path.splitext(filename)[0]
    vad_log_path = os.path.join(path_vad, f"{basename}.srt")
    """
        if os.path.exists(vad_log_path):
        print('VAD记录存在，跳过')
        continue
    """


    vad_result = vad_pipeline(audio_path)
    srt = pysrt.SubRipFile()

    for idx, (segment, _, score) in enumerate(vad_result.itertracks(yield_label=True), start=1):
        sub_item = pysrt.SubRipItem(
            index=idx,
            start=pysrt.SubRipTime.from_ordinal(int(segment.start * 1000)),
            end=pysrt.SubRipTime.from_ordinal(int(segment.end * 1000)),
            text=f"默认文本{idx}"
        )
        srt.append(sub_item)

    srt.save(vad_log_path)
    print(f"VAD记录写入: {vad_log_path}")

# 6生成字幕

In [ ]:
import os
import pysrt
import faster_whisper.vad
import faster_whisper.transcribe  # 必须显式导入，以便 patch 定位
from unittest.mock import patch
from faster_whisper import WhisperModel
#############################################################################################################################
print('设备:', device, '类型:', compute_type)

def get_custom_vad_patch(audio, vad_parameters=None):
    """
    这个函数会被注入到 faster_whisper.transcribe 内部。
    """
    # 打印测试：如果补丁生效，你会看到这行输出
    if hasattr(faster_whisper.vad, "_custom_segments"):
        segment_count = len(faster_whisper.vad._custom_segments)
        print(f"  [Patch Success] 补丁函数触发：已注入来自 SRT 的 {segment_count} 个片段")
        return faster_whisper.vad._custom_segments

    print("  [Patch Warning] 补丁函数已触发，但未发现预设片段数据！")
    return []
model = WhisperModel(
    model_size_or_path=asr_model,
    device=device,
    compute_type=compute_type,
    download_root=path_model
)

for filename in os.listdir(path_audio):
    if not filename.endswith((".wav", ".mp3", ".flac")):
        continue

    audio_path = os.path.join(path_audio, filename)
    basename = os.path.splitext(filename)[0]
    vad_log_path = os.path.join(path_vad, f"{basename}.srt")
    asr_log_path = os.path.join(path_asr, f"{basename}.srt")

    if not os.path.exists(vad_log_path):
        print(f"跳过：找不到 VAD 字幕文件 {vad_log_path}")
        continue

    # 1. 解析 SRT 得到采样点区间
    vad_subs = pysrt.open(vad_log_path)
    custom_chunks = []
    for sub in vad_subs:
        custom_chunks.append({
            'start': int(sub.start.ordinal / 1000 * 16000),
            'end': int(sub.end.ordinal / 1000 * 16000)
        })

    # 将数据存入模块变量，供补丁函数读取
    faster_whisper.vad._custom_segments = custom_chunks
    print(f"\n开始推理: {audio_path}")
    with patch("faster_whisper.transcribe.get_speech_timestamps", side_effect=get_custom_vad_patch):

        segments, _ = model.transcribe(
            audio=audio_path,
            language=asr_language,
            task="translate",
            vad_filter=True,  # 必须为 True 才会触发调用
            condition_on_previous_text=True,
            log_progress=True
        )

        asr_subs = pysrt.SubRipFile()

        # 3. 迭代处理流式输出的结果
        for segment in segments:
            text = segment.text.strip()
            # 创建新的 SRT 条目
            new_sub = pysrt.SubRipItem(
                index=len(asr_subs) + 1,
                start=pysrt.SubRipTime.from_ordinal(int(segment.start * 1000)),
                end=pysrt.SubRipTime.from_ordinal(int(segment.end * 1000)),
                text=text
            )
            asr_subs.append(new_sub)
            print(f"[{segment.start:.2f}s -> {segment.end:.2f}s]: {text}")
            # 保存 ASR 结果
            asr_subs.save(asr_log_path, encoding='utf-8')


    print(f"处理完成，保存至: {asr_log_path}")